<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Rotation Representations

In [Notebook 2](./2-imu-estimation-and-drift.ipynb) a fixed orientation was assumed while investigating position drift. When the Duckiedrone tilts, however, a vector expressed along its body axes must be related to the stationary world axes before it can describe motion in the room.

This notebook addresses the following questions:

- How do the body and world frames define the Duckiedrone's attitude, and how are vector components related between them? 

- What are the typical mathematical representations and conventions used, and why are Euler angles somewhat limited?

- What is gimbal lock, and does it restrict the Duckiedrone's physical motion?

## Euler angles and the rotation matrix

Choose a __body frame__, labeled $\mathrm{B}$, fixed to the Duckiedrone with its origin at the center of mass. Its $x$ axis points toward the front of the Duckiedrone, its $y$ axis toward the left, and its $z$ axis toward the top. These directions are relative to the Duckiedrone: when it tilts, its body $z$ axis tilts too.

Let the __world frame__, labeled $\mathrm{W}$, remain fixed with respect to the room. Its $z$ axis points upward, opposite the direction of local gravitational acceleration. A fixed horizontal direction defines its $x$ axis, and the $y$ axis completes a [right-handed reference frame](https://en.wikipedia.org/wiki/Right-hand_rule). At zero reference attitude, the body and world axes coincide.

The orientation, or attitude, of the Duckiedrone describes how its body axes are turned relative to the world axes. Only orientation matters here, so both frames share one origin. A different origin would also need a translation, as detailed in [this Duckietown lecture on representations](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/models/01-representations.html).

### Mathematical notations

[Table 1](#table-1) collects the notation used in this notebook.

<a id="table-1"></a>

Table 1: Summary of symbols used for rotations in this notebook.

| Symbol | Meaning | Units |
| --- | --- | --- |
| $\mathbf{R}_{\mathrm{WB}}$ | Rotation matrix mapping body-frame components into world-frame components | Dimensionless |
| $\mathbf{p}_{\mathrm{B}}$, $\mathbf{p}_{\mathrm{W}}$ | Coordinates of the same point in the body and world frames, with a shared origin | $\mathrm{m}$ |
| $\tilde{\mathbf{f}}_{\mathrm{B}}$, $\tilde{\mathbf{f}}_{\mathrm{W}}$ | Measured proper acceleration in body-frame and world-frame components | $\mathrm{m/s^2}$ |
| $\alpha$, $\beta$, $\gamma$ | Roll, pitch, and yaw in the ZYX convention defined below | $\mathrm{rad}$ |
| $\mathbf{e}_x$, $\mathbf{e}_y$, $\mathbf{e}_z$ | Cartesian unit column vectors | Dimensionless |
| $\mathbf{I}$, $(\cdot)^{\mathsf{T}}$ | Three-by-three identity matrix and transpose notation | - |

Given an orientation, let $\mathbf{R}_{\mathrm{WB}} \in \mathbb{R}^{3 \times 3}$ be the matrix representing the body frame to world frame rotation, such that the coordinates of the same point $\mathbf{p}_{\mathrm{(\cdot)}} \in \mathbb{R}^3$ expressed in the two frames are related by:

<span id="equation-1"></span>

$$
\mathbf{p}_{\mathrm{W}} = \mathbf{R}_{\mathrm{WB}}\,\mathbf{p}_{\mathrm{B}}.
\tag{1}
$$

The subscripts indicate the direction of the mapping: body-frame components enter the calculation, and world-frame components come out. The point remains the same; the coordinates describe it along different axes.

A valid rotation matrix satisfies:

$$
\mathbf{R}_{\mathrm{WB}}^{\mathsf{T}}\mathbf{R}_{\mathrm{WB}} = \mathbf{I},
\qquad
\det\!\left(\mathbf{R}_{\mathrm{WB}}\right) = +1.
$$

The first condition makes the columns unit length and mutually perpendicular. The second preserves handedness and rules out a reflection. Together, these conditions ensure that the transformation preserves lengths and angles. The inverse mapping uses the transpose:

$$
\mathbf{p}_{\mathrm{B}} = \mathbf{R}_{\mathrm{WB}}^{\mathsf{T}}\mathbf{p}_{\mathrm{W}}.
$$

__Euler angles__ describe orientation through three rotations: roll $\alpha$ about an $x$ axis, pitch $\beta$ about a $y$ axis, and yaw $\gamma$ about a $z$ axis. Their signs are defined positive according to [the right-hand rule for rotations](https://en.wikipedia.org/wiki/Right-hand_rule).  The physical meaning of Euler angles depends on the order of the rotations they represent, and on whether the axes remain fixed or turn with the Duckiedrone. Angles in the equations are measured in radians ($\mathrm{rad}$), while degrees ($^\circ$) are typically used in displayed examples, as they are easier to interpret.

[Figure 1](#figure-1) shows the three axes at the aligned reference orientation.

<a id="figure-1"></a>

<p class="lx-figure" style="width:500px;">
  <img src="../assets/_images/duckiedrone_roll_pitch_yaw.jpg" alt="Roll, pitch, and yaw axes on a level Duckiedrone." style="width:100%;" />
</p>

Figure 1: The roll, pitch, and yaw axes at the aligned reference orientation. The body and world axes coincide in this drawing.

### One rotation at a time

Write $\mathbf{R}_i(\theta)$ for a right-handed rotation through angle $\theta$ about axis $i \in \{x, y, z\}$, as in the [notation reference](../assets/data/notation.md).

Starting from the aligned reference orientation, a yaw by angle $\gamma$ has rotation matrix:

<span id="equation-2"></span>

$$
\mathbf{R}_z(\gamma) =
\begin{bmatrix}
\cos(\gamma) & -\sin(\gamma) & 0 \\
\sin(\gamma) & \cos(\gamma) & 0 \\
0 & 0 & 1
\end{bmatrix}.
\tag{2}
$$

The $z$ coordinate is unchanged because this rotation turns about the $z$ axis, while the horizontal coordinates turn as a point on a circle does.

The nose direction in body components is the unit vector $\mathbf{e}_x = [1, 0, 0]^{\mathsf{T}}$. For a yaw of $90^\circ$, substituting $\cos(90^\circ)=0$ and $\sin(90^\circ)=1$ into [Equation (2)](#equation-2) gives:

<span id="equation-3"></span>

$$
\mathbf{R}_z(90^\circ)\,\mathbf{e}_x =
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
= \mathbf{e}_y.
\tag{3}
$$

The resulting components describe a nose pointing along the world $+y$ axis: a quarter turn to the left from the reference orientation.

The pitch matrix instead leaves the $y$ coordinate unchanged:

$$
\mathbf{R}_y(\beta) =
\begin{bmatrix}
\cos(\beta) & 0 & \sin(\beta) \\
0 & 1 & 0 \\
-\sin(\beta) & 0 & \cos(\beta)
\end{bmatrix}.
$$

With $y$ pointing left, the right-hand rule makes positive pitch tip the nose down. In particular, $\mathbf{R}_y(90^\circ)\mathbf{e}_x=-\mathbf{e}_z$, while a pitch of $-90^\circ$ points the nose upward.

The roll matrix, finally, leaves the $x$ coordinate unchanged:

$$
\mathbf{R}_x(\alpha) =
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos(\alpha) & -\sin(\alpha) \\
0 & \sin(\alpha) & \cos(\alpha)
\end{bmatrix}.
$$

A positive roll raises the Duckiedrone's left side from the reference orientation. For example, $\mathbf{R}_x(90^\circ)\mathbf{e}_y=\mathbf{e}_z$: the body $+y$ direction turns toward world $+z$.

### Putting the three together

The ZYX convention used in this notebook combines the elementary rotations as:

<span id="equation-4"></span>

$$
\mathbf{R}_{\mathrm{WB}}
=
\mathbf{R}_z(\gamma)\,\mathbf{R}_y(\beta)\,\mathbf{R}_x(\alpha).
\tag{4}
$$

Read about the fixed world axes, the rightmost matrix acts first: roll, then pitch, then yaw. Read about the Duckiedrone's own axes, the same product is yaw, then pitch, then roll.

Both readings produce [Equation (4)](#equation-4). Stating the axes and their order makes the convention unambiguous; the name alone is insufficient because other sources can use different conventions.

The order matters because matrix multiplication does not generally commute. A roll of $90^\circ$ followed by a pitch of $90^\circ$, both about fixed world axes, leaves the nose pointing down. Reversing those two rotations leaves it pointing along world $+y$.

### The rotation matrix is a representation too

The nine entries of $\mathbf{R}_{\mathrm{WB}}$ describe orientation directly. Its first column is $\mathbf{R}_{\mathrm{WB}}\mathbf{e}_x$, the direction of the body $x$ axis expressed in world components. Its second and third columns similarly describe the body $y$ and $z$ axes.

For the yawed Duckiedrone in [Equation (3)](#equation-3), the first column of $\mathbf{R}_z(90^\circ)$ is $\mathbf{e}_y$, matching the nose direction derived there.

The same mapping applies to a measured IMU vector whose components are already expressed in the body frame. For example:

$$
\tilde{\mathbf{f}}_{\mathrm{W}}
=
\mathbf{R}_{\mathrm{WB}}\,\tilde{\mathbf{f}}_{\mathrm{B}}.
$$

This operation expresses the same proper-acceleration measurement along the world axes. It does not remove gravity: recovering linear acceleration still requires the gravity term in [Equation (1) in Notebook 2](./2-imu-estimation-and-drift.ipynb#equation-1).

Although a rotation matrix contains nine entries, its columns must remain unit length, mutually perpendicular, and right-handed. These constraints leave three independent degrees of freedom, matching the three needed to describe orientation. The matrix remains a valid representation even at orientations where Euler angles become ambiguous.

## Gimbal lock

[Figure 2](#figure-2) illustrates a related difficulty with the angles used to locate a point on a sphere.

<a id="figure-2"></a>

<p class="lx-figure" style="width:250px;">
  <img src="../assets/_images/3d_unit_sphere.jpg" alt="Longitude and latitude angles locating a point on a sphere." style="width:100%;" />
</p>

Figure 2: Longitude $\theta$ is measured around the equator, and latitude $\phi$ is measured up from the equator. At either pole, every longitude identifies the same point.

Consider any point on a sphere. Two angles, longitude and latitude, are sufficient to identify it, as on a globe. The trouble is at the north pole, where the latitude is $90^\circ$ and every longitude gives the same point. Someone walking straight over the pole never changes direction, yet their longitude jumps by $180^\circ$ the moment they cross. The path remains smooth, while longitude becomes undefined at the pole and changes its coordinate value discontinuously across it.

Euler angles break in the same way when pitch reaches $\beta = \pm 90^\circ$. 

At $\beta=-90^\circ$, the Duckiedrone's nose points upward and its body $x$ axis aligns with world $+z$. Roll and yaw then act about the same axis in the same direction, so the orientation depends on their sum, $\alpha+\gamma$. Increasing roll and decreasing yaw by the same amount leaves the orientation unchanged.

At $\beta=+90^\circ$, the nose points downward and the corresponding axes point in opposite directions. The orientation then depends on $\alpha-\gamma$, so increasing roll and yaw by the same amount leaves it unchanged.

[Figure 3](#figure-3) compares the aligned reference orientation with the nose-up case.

<a id="figure-3"></a>

<p class="lx-figure" style="width:760px;">
  <img src="../assets/_images/duckiedrone_pitch_0.jpg" alt="Level Duckiedrone with its nose along world x." style="width:48%; display:inline-block;" />
  <img src="../assets/_images/duckiedrone_pitch_90.jpg" alt="Duckiedrone pitched nose-up, with its nose along world z." style="width:48%; display:inline-block;" />
</p>

Figure 3: The black axes remain fixed in the world. Left: the body and world axes coincide. Right: at $\beta=-90^\circ$, the body $x$ axis points along world $+z$, so roll and yaw act about the same vertical line. The blue marker identifies the body $+y$ side.

At either singular orientation, small roll and yaw changes produce dependent effects. Pitch still tips the nose toward world $x$, but no small change of the three angles tips it sideways, toward world $y$. To tilt the nose sideways by just $0.001\,\mathrm{rad}$, yaw has to jump by $90^\circ$, and roll by $90^\circ$ the other way to keep the rest of the orientation in place. A small physical change in orientation can therefore require large, discontinuous changes in the Euler angles.

The term _gimbal lock_ comes from a mechanical gimbal, a set of nested rings with separate rotation axes. When two ring axes align, the mechanism loses an independent instantaneous rotation direction. The video [Euler (gimbal lock) Explained](https://youtu.be/zc8b2Jo7mno?t=39) illustrates that alignment. The Euler-angle singularity takes its name from this mechanism.

## Activity 1: watch gimbal lock happen

<a id="activity-1"></a>

The example below applies [Equation (4)](#equation-4) to a toy airplane. The red dot marks the nose, and the blue dot marks the wing tip on the body $+y$ side, corresponding to the blue marker in [Figure 3](#figure-3). Gray arrows show the fixed world axes; orange arrows show the rotating body axes.

1. Start with all angles at zero. Set yaw to $90^\circ$ and compare the nose direction with [Equation (3)](#equation-3), then return yaw to zero.

2. Set pitch to $-90^\circ$. Predict the final orientation after setting roll to $30^\circ$ and yaw to $-30^\circ$, then apply those changes. Compare the result with the initial nose-up orientation and relate it to $\alpha+\gamma$.

3. Reset roll and yaw to zero, then set pitch to $+90^\circ$. Set roll to $30^\circ$ and yaw to $30^\circ$. Compare the result with the initial nose-down orientation and relate it to $\alpha-\gamma$.

4. Keep roll and yaw at $30^\circ$ and reduce pitch to $80^\circ$. Compare this orientation with roll and yaw both set to zero at the same pitch. The equality observed at $90^\circ$ no longer holds.

5. Restore all three angles to zero.

At exactly $\pm90^\circ$ pitch, the readout identifies the combination of roll and yaw that determines the orientation. This exposes the dependence between the angle coordinates while the rotation matrix continues to describe the airplane's orientation.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatSlider, interact
from mpl_toolkits.mplot3d import Axes3D

# Symbols follow the notebook:
#   roll, pitch, yaw   The angles alpha, beta, and gamma, in degrees for the sliders.
#   a, b, g            The same three angles in radians, as used in the matrices.

# A toy airplane: nose first, then fuselage, tail fin, and the two wings.
airplane = np.array(
    (
        np.concatenate(
            (
                1.15,
                np.linspace(1, -0.75, 25),
                np.linspace(-0.75, -1, 15),
                np.linspace(0.4, 0, 10),
                np.linspace(0.4, 0, 10),
            ),
            axis=None,
        ),
        np.concatenate(
            (
                0,
                np.linspace(0, 0, 25),
                np.linspace(0, 0, 15),
                np.linspace(0, 0.9, 10),
                np.linspace(0, -0.9, 10),
            ),
            axis=None,
        ),
        np.concatenate(
            (
                0,
                np.linspace(0, 0, 25),
                np.linspace(0, 0.5, 15),
                np.linspace(0, 0, 10),
                np.linspace(0, 0, 10),
            ),
            axis=None,
        ),
    )
)
WING_TIP = np.argmax(airplane[1])  # The tip of the wing on the +y side.


def rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    """Build R = RZ @ RY @ RX, the ZYX convention from the section above."""
    a, b, g = np.deg2rad([roll_deg, pitch_deg, yaw_deg])
    rx = np.array(
        [[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]]
    )
    ry = np.array(
        [[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]]
    )
    rz = np.array(
        [[np.cos(g), -np.sin(g), 0], [np.sin(g), np.cos(g), 0], [0, 0, 1]]
    )
    return rz @ ry @ rx


def show(roll=0.0, pitch=0.0, yaw=0.0):
    R_WB = rotation_matrix(roll, pitch, yaw)  # noqa: N806
    pts = R_WB @ airplane

    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(projection=Axes3D.name)
    ax.view_init(elev=10, azim=30)

    for i, axis_name in enumerate("xyz"):
        world_axis = np.eye(3)[:, i]
        body_axis = R_WB[:, i]
        ax.quiver(
            0, 0, 0, *(1.4 * world_axis),
            color="0.6", arrow_length_ratio=0.08,
        )
        ax.text(*(1.5 * world_axis), f"{axis_name}_W", color="0.4")
        ax.quiver(
            0, 0, 0, *(0.75 * body_axis),
            color="tab:orange", arrow_length_ratio=0.12,
        )
        ax.text(*(0.85 * body_axis), f"{axis_name}_B", color="tab:orange")

    ax.scatter(pts[0, 1:], pts[1, 1:], pts[2, 1:], c="black", s=60)
    ax.scatter(pts[0, 0], pts[1, 0], pts[2, 0], c="red", s=120)
    ax.scatter(*pts[:, WING_TIP], c="tab:blue", s=120)

    for lim in (ax.set_xlim3d, ax.set_ylim3d, ax.set_zlim3d):
        lim(-1.6, 1.6)
    ax.set_box_aspect((1, 1, 1))

    if np.isclose(abs(pitch), 90.0, rtol=0.0, atol=1e-9):
        # Nose up: the two angles add; nose down: they subtract.
        op, locked = ("+", roll + yaw) if pitch < 0 else ("-", roll - yaw)
        ax.set_title(
            f"Euler-angle singularity at pitch {pitch:+.0f} deg\n"
            f"only roll {op} yaw matters, and it is {locked:+.0f} deg",
            color="tab:red",
        )
    else:
        ax.set_title(
            f"roll {roll:+.0f} deg   pitch {pitch:+.0f} deg   yaw {yaw:+.0f} deg"
        )

    fig.tight_layout()
    plt.show()


interact(
    show,
    roll=FloatSlider(
        value=0.0,
        min=-180,
        max=180,
        step=1,
        description="roll [deg]",
        style={"description_width": "100px"},
        continuous_update=False,
    ),
    pitch=FloatSlider(
        value=0.0,
        min=-90,
        max=90,
        step=1,
        description="pitch [deg]",
        style={"description_width": "100px"},
        continuous_update=False,
    ),
    yaw=FloatSlider(
        value=0.0,
        min=-180,
        max=180,
        step=1,
        description="yaw [deg]",
        style={"description_width": "100px"},
        continuous_update=False,
    ),
);


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

## Summary and next steps

A rotation matrix relates vector components in two frames, with its columns giving the body axes in world components. Roll, pitch, and yaw provide another description, but their meaning depends on the stated axes and rotation order.

At ZYX pitch $\pm90^\circ$, roll and yaw become locally dependent. The rotation matrix remains well defined, and the Duckiedrone retains its ability to rotate. [Notebook 4](./4-quaternions-and-rotation-representations.ipynb) introduces unit quaternions, which represent orientation without this Euler-angle singularity.

## Further reading

[Three-dimensional linear transformations](https://www.youtube.com/watch?v=rHLEWRxRGiM) is a nice video illustrating the relationship between matrix columns and transformed axes. The Duckietown [Representations - Part 1](https://hubs.ly/Q02bk6-V0) slides and [associated lecture video](https://vimeo.com/588472832?fl=pl&fe=sh) develop rotation representations, while Chapter 3 of [Modern Robotics](https://hades.mech.northwestern.edu/index.php/Modern_Robotics) covers rotation matrices and frame transformations.

[Gimbal lock](https://en.wikipedia.org/wiki/Gimbal_lock) gives additional background on aligned rotation axes.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()